In [15]:
# ETS Model with Store Location and Holiday Flag, Validated on Test Set
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt

# Load data
train = pd.read_csv('./sfo/train.csv')
test = pd.read_csv('./sfo/test.csv')

In [16]:
# Display binary, categorical, and numeric columns, and which can be binary encoded or used for PCA
import pandas as pd
import numpy as np

# Exclude target column from feature analysis
target_col = 'Category' if 'Category' in train.columns else train.columns[-1]
feature_cols = [col for col in train.columns if col != target_col]

# Identify binary columns (2 unique values, not the target)
binary_cols = [col for col in feature_cols if train[col].nunique(dropna=False) == 2]

# Identify categorical columns (object or category dtype, more than 2 unique values)
cat_cols = [col for col in feature_cols if train[col].dtype in ['object', 'category'] and train[col].nunique() > 2]

# Identify numeric columns (excluding binary and target)
numeric_cols = [col for col in feature_cols if np.issubdtype(train[col].dtype, np.number) and col not in binary_cols]

# Columns suitable for PCA: numeric columns with >2 unique values
pca_candidates = [col for col in numeric_cols if train[col].nunique() > 2]

print('Binary columns (can be binary encoded):', binary_cols)
print('Categorical columns (can be one-hot or label encoded):', cat_cols)
print('Numeric columns:', numeric_cols)
print('Columns suitable for PCA:', pca_candidates)

Binary columns (can be binary encoded): []
Categorical columns (can be one-hot or label encoded): ['Dates', 'Descript', 'DayOfWeek', 'PdDistrict', 'Resolution', 'Address']
Numeric columns: ['X', 'Y']
Columns suitable for PCA: ['X', 'Y']


In [34]:
# Label encode categorical columns and scale numeric columns (excluding target), then split train.csv for 80/20 train/val split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

# Identify columns
target_col = 'Category' if 'Category' in train.columns else train.columns[-1]
feature_cols = [col for col in train.columns if col != 'Category' and col != 'Dates' and col != 'Address' and col != 'Descript' and col != 'Resolution']
cat_cols = [col for col in feature_cols if train[col].dtype in ['object', 'category'] and train[col].nunique() > 2]
numeric_cols = [col for col in feature_cols if np.issubdtype(train[col].dtype, np.number)]

# Label encode categorical columns
X_cat = np.empty((len(train), 0))
if cat_cols:
    X_cat = np.zeros((len(train), len(cat_cols)))
    for i, col in enumerate(cat_cols):
        le = LabelEncoder()
        X_cat[:, i] = le.fit_transform(train[col].astype(str))

# Scale numeric columns
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(train[numeric_cols]) if numeric_cols else np.empty((len(train),0))

# Combine features
from numpy import hstack
X_all = hstack([X_cat, X_num_scaled]) if X_cat.shape[1] > 0 else X_num_scaled
y = train[target_col]

# 80/20 split on train data
X_train, X_val, y_train, y_val = train_test_split(X_all, y, test_size=0.2, random_state=42, stratify=y)

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)

X_train shape: (702439, 4)
X_val shape: (175610, 4)


In [35]:
# Label encode categorical columns and scale numeric columns for test.csv (skip 'Dates', use encoders/scaler from train), do not split
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Use cat_cols and numeric_cols from previous cell (fitted on train), skip 'Dates' column
test_feature_cols = [col for col in test.columns if col != 'Category' and col != 'Dates' and col != 'Address']
test_cat_cols = [col for col in cat_cols if col in test_feature_cols]
test_num_cols = [col for col in numeric_cols if col in test_feature_cols]

test_cat = np.empty((len(test), 0))
if test_cat_cols:
    test_cat = np.zeros((len(test), len(test_cat_cols)))
    for i, col in enumerate(test_cat_cols):
        le = LabelEncoder()
        le.fit(train[col].astype(str))  # Fit on train to ensure same mapping
        test_cat[:, i] = le.transform(test[col].astype(str))

test_num_scaled = scaler.transform(test[test_num_cols]) if 'scaler' in locals() and test_num_cols else np.empty((len(test),0))

from numpy import hstack
test_all = hstack([test_cat, test_num_scaled]) if test_cat.shape[1] > 0 else test_num_scaled

print('test_all shape:', test_all.shape)

test_all shape: (884262, 4)


In [36]:
# Train SVM model and print classification metrics (F1, accuracy, precision, recall, confusion matrix, etc.)
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
import numpy as np

# Fit SVM model
svm = LinearSVC(max_iter=2000, random_state=42)
svm.fit(X_train, y_train)

# Predict on validation set
y_pred = svm.predict(X_val)

# Metrics
f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
acc = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_val, y_pred, average='weighted', zero_division=0)
cm = confusion_matrix(y_val, y_pred)

print(f"F1 Score: {f1:.3f}")
print(f"Accuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))

F1 Score: 0.066
Accuracy: 0.199
Precision: 0.040
Recall: 0.199

Confusion Matrix:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Classification Report:
                             precision    recall  f1-score   support

                      ARSON       0.00      0.00      0.00       303
                    ASSAULT       0.00      0.00      0.00     15375
                 BAD CHECKS       0.00      0.00      0.00        81
                    BRIBERY       0.00      0.00      0.00        58
                   BURGLARY       0.00      0.00      0.00      7351
         DISORDERLY CONDUCT       0.00      0.00      0.00       864
DRIVING UNDER THE INFLUENCE       0.00      0.00      0.00       454
              DRUG/NARCOTIC       0.00      0.00      0.00     10794
                DRUNKENNESS       0.00      0.00      0.00       856
               EMBEZZLEMENT       0.00      0.00      0.00       233
              

In [37]:
# Predict Category for test_all using SVM and save only Id and predicted_Category to CSV
test_pred = svm.predict(test_all)
test_results = test['Id'].copy()
test_results['Category'] = test_pred
test_results.to_csv('./sfo/svm_test_predictions.csv', index=False)


In [39]:
# Train Random Forest model and print classification metrics (F1, accuracy, precision, recall, confusion matrix, etc.)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)

# Predict on validation set
y_pred_rf = rf_clf.predict(X_val)

# Metrics
f1_rf = f1_score(y_val, y_pred_rf, average='weighted', zero_division=0)
acc_rf = accuracy_score(y_val, y_pred_rf)
precision_rf = precision_score(y_val, y_pred_rf, average='weighted', zero_division=0)
recall_rf = recall_score(y_val, y_pred_rf, average='weighted', zero_division=0)
cm_rf = confusion_matrix(y_val, y_pred_rf)

print(f"Random Forest F1 Score: {f1_rf:.3f}")
print(f"Random Forest Accuracy: {acc_rf:.3f}")
print(f"Random Forest Precision: {precision_rf:.3f}")
print(f"Random Forest Recall: {recall_rf:.3f}")
print("\nRandom Forest Confusion Matrix:")
print(cm_rf)
print("\nRandom Forest Classification Report:")
print(classification_report(y_val, y_pred_rf, zero_division=0))

Random Forest F1 Score: 0.234
Random Forest Accuracy: 0.262
Random Forest Precision: 0.223
Random Forest Recall: 0.262

Random Forest Confusion Matrix:
[[   2   38    0 ...   26    5    5]
 [   7 3059    1 ...  557  280   72]
 [   0    2    0 ...    4    0    1]
 ...
 [  14  680    0 ... 2635  147   32]
 [   4  756    1 ...  209  324   18]
 [   1  303    0 ...   50   43   55]]

Random Forest Classification Report:
                             precision    recall  f1-score   support

                      ARSON       0.02      0.01      0.01       303
                    ASSAULT       0.19      0.20      0.19     15375
                 BAD CHECKS       0.00      0.00      0.00        81
                    BRIBERY       0.00      0.00      0.00        58
                   BURGLARY       0.14      0.11      0.12      7351
         DISORDERLY CONDUCT       0.05      0.01      0.02       864
DRIVING UNDER THE INFLUENCE       0.02      0.00      0.01       454
              DRUG/NARCOTIC  

In [41]:
# Predict Category for test_all using Random Forest and save only Id and predicted_Category to CSV
test_pred = rf_clf.predict(test_all)
test_results = test['Id'].copy()
test_results['Category'] = test_pred
test_results.to_csv('./sfo/rf_test_predictions.csv', index=False)

In [45]:
# Train Gradient Boosting model and print classification metrics (F1, accuracy, precision, recall, confusion matrix, etc.)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)

# Predict on validation set
y_pred_gb = gb_clf.predict(X_val)

# Metrics
f1_gb = f1_score(y_val, y_pred_gb, average='weighted', zero_division=0)
acc_gb = accuracy_score(y_val, y_pred_gb)
precision_gb = precision_score(y_val, y_pred_gb, average='weighted', zero_division=0)
recall_gb = recall_score(y_val, y_pred_gb, average='weighted', zero_division=0)
cm_gb = confusion_matrix(y_val, y_pred_gb)

print(f"Gradient Boosting F1 Score: {f1_gb:.3f}")
print(f"Gradient Boosting Accuracy: {acc_gb:.3f}")
print(f"Gradient Boosting Precision: {precision_gb:.3f}")
print(f"Gradient Boosting Recall: {recall_gb:.3f}")
print("\nGradient Boosting Confusion Matrix:")
print(cm_gb)
print("\nGradient Boosting Classification Report:")
print(classification_report(y_val, y_pred_gb, zero_division=0))

Gradient Boosting F1 Score: 0.161
Gradient Boosting Accuracy: 0.243
Gradient Boosting Precision: 0.230
Gradient Boosting Recall: 0.243

Gradient Boosting Confusion Matrix:
[[  1   8   0 ...   0   0   0]
 [  2 369   4 ...   2   0   1]
 [  0   0   0 ...   0   0   0]
 ...
 [  6 104   0 ...  32   1   0]
 [  1  89   0 ...   3   1   0]
 [  1  57   0 ...   1   0   1]]

Gradient Boosting Classification Report:
                             precision    recall  f1-score   support

                      ARSON       0.02      0.00      0.01       303
                    ASSAULT       0.19      0.02      0.04     15375
                 BAD CHECKS       0.00      0.00      0.00        81
                    BRIBERY       0.00      0.00      0.00        58
                   BURGLARY       0.09      0.03      0.04      7351
         DISORDERLY CONDUCT       0.09      0.01      0.02       864
DRIVING UNDER THE INFLUENCE       0.00      0.00      0.00       454
              DRUG/NARCOTIC       0.30   

In [46]:
# Predict Category for test_all using Gradient Boosting and save only Id and predicted_Category to CSV
test_pred = gb_clf.predict(test_all)
test_results = test['Id'].copy()
test_results['Category'] = test_pred
test_results.to_csv('./sfo/gb_test_predictions.csv', index=False)

In [43]:
# Train Extra Trees model (fast tree-based) and print classification metrics (F1, accuracy, precision, recall, confusion matrix, etc.)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

et_clf = ExtraTreesClassifier(random_state=42, n_jobs=-1)
et_clf.fit(X_train, y_train)

# Predict on validation set
y_pred_et = et_clf.predict(X_val)

# Metrics
f1_et = f1_score(y_val, y_pred_et, average='weighted', zero_division=0)
acc_et = accuracy_score(y_val, y_pred_et)
precision_et = precision_score(y_val, y_pred_et, average='weighted', zero_division=0)
recall_et = recall_score(y_val, y_pred_et, average='weighted', zero_division=0)
cm_et = confusion_matrix(y_val, y_pred_et)

print(f"Extra Trees F1 Score: {f1_et:.3f}")
print(f"Extra Trees Accuracy: {acc_et:.3f}")
print(f"Extra Trees Precision: {precision_et:.3f}")
print(f"Extra Trees Recall: {recall_et:.3f}")
print("\nExtra Trees Confusion Matrix:")
print(cm_et)
print("\nExtra Trees Classification Report:")
print(classification_report(y_val, y_pred_et, zero_division=0))

Extra Trees F1 Score: 0.232
Extra Trees Accuracy: 0.264
Extra Trees Precision: 0.225
Extra Trees Recall: 0.264

Extra Trees Confusion Matrix:
[[   2   56    0 ...   14    3    4]
 [  26 4009    2 ...  384  152   38]
 [   0    6    0 ...    3    0    0]
 ...
 [  25 1048    2 ... 2026   69    8]
 [   9  998    2 ...  143  217   15]
 [   3  392    0 ...   36   22   34]]

Extra Trees Classification Report:
                             precision    recall  f1-score   support

                      ARSON       0.01      0.01      0.01       303
                    ASSAULT       0.18      0.26      0.21     15375
                 BAD CHECKS       0.00      0.00      0.00        81
                    BRIBERY       0.00      0.00      0.00        58
                   BURGLARY       0.13      0.14      0.14      7351
         DISORDERLY CONDUCT       0.04      0.02      0.02       864
DRIVING UNDER THE INFLUENCE       0.03      0.01      0.02       454
              DRUG/NARCOTIC       0.29   

In [44]:
# Predict Category for test_all using Extra tree and save only Id and predicted_Category to CSV
test_pred = et_clf.predict(test_all)
test_results = test['Id'].copy()
test_results['Category'] = test_pred
test_results.to_csv('./sfo/et_test_predictions.csv', index=False)